**### ** Bronzework Incremental****

## **Step1 - Import and Setup**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog databricks_dev ")

In [0]:
spark.sql("create schema if not exists 00_bronze")


### **Step 2 - Bronze control table**
 This table stores the **watermark** for each source table.
-  the latest timestamp already processed
- the latest primary key processed at that timestamp
- how many rows were written in the latest run

This is what makes the Bronze load incremental and rerun safe.

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS 00_bronze.ingestion_control (
    layer STRING,
    table_name STRING,
    ts_col STRING,
    pk_col STRING,
    last_successful TIMESTAMP,
    last_successful_pk BIGINT,
    last_run_id STRING,
    rows_written LONG,
    run_status STRING,
    updated_ts TIMESTAMP
) USING DELTA
""")

**Step 3 - Source table configuration**
This cell defines which source tables will be loaded into bronze and which columns should be used as:
- primary key
- timestamp/watermark column

> It will also creates a unique **bronze_run_id** for the current pipeline run

In [0]:
tables_config= {
    "orders": { "ts_col": "updated_at", "pk_col": "order_id"},
    "products": { "ts_col": "updated_at","pk_col": "product_id"},
     "payments": {"ts_col": "processed_at","pk_col": "payment_id"}           
    }

bronze_run_id=str(uuid.uuid4())
print(f"bronze run id: {bronze_run_id}")


    

**### Step 4 - Helper functions**                                                  
This cell contains reusable functions:
- **get_last_successful_watermark**() reads the last processed timestamp/watermark from the control 
- **upsert_bronze_control()** updates the control table after a successful Bronze 
load

These functions keep the main logic cleaner and easier to underatnd.

In [0]:
def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("00_bronze.ingestion_control")
        .filter(F.col("layer") == "bronze")
        .filter(F.col("table_name") == table_name)
        .filter(F.col("run_status") == "success")
        .orderBy(F.col("updated_ts").desc())
        .limit(1)
        .select(
            F.col("last_successful").alias("last_successful_watermark"),
            F.col("last_successful_pk")
        )
    )
    rows = ctrl.collect()
    if len(rows) == 0:
        return None, None
    else:
        return rows[0]["last_successful_watermark"], rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    control_df = (spark.createDataFrame(
        [("bronze",table_name, ts_col, pk_col, last_ts, int(last_pk) if last_pk is not None else None, run_id, int(rows_written), "success", datetime.utcnow()
        )],
        schema="""layer STRING, table_name STRING, ts_col STRING, pk_col STRING, last_successful TIMESTAMP, last_successful_pk BIGINT, last_run_id STRING, rows_written LONG, run_status STRING, updated_ts TIMESTAMP""")
    )
    DeltaTable.forName(spark, "00_bronze.ingestion_control").alias("t").merge(
        control_df.alias("s"), "t.table_name = s.table_name AND t.layer = s.layer"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    




**### Step 5 - Bronze incremental load loop**
This is the main Bronze logic.

For each table, the notebook:

- Reads the latest timestamp/watermark
- reads the source SQl table
- filters only **new/changed rows**
- adds Bronze audit columns
- appends the rows into the Bronze Delta table
- updates the control table

This is the core incremental loading logic

In [0]:
for table_name, table_config in tables_config.items():
    ts_col = table_config["ts_col"]
    pk_col = table_config["pk_col"]
    source_table=f"`azuresqlserver-connection_catalog`.dbo.{table_name}"
    target_table=f"databricks_dev.`00_bronze`.{table_name}_raw"
    last_successful, last_successful_pk = get_last_successful_watermark(table_name)

    print(f"\n===processing table {table_name}===")
    print(f"last_ts: {last_successful}, last_pk: {last_successful_pk}")
    
    source_df = spark.read.table(source_table).withColumn(ts_col, F.col(ts_col).cast("timestamp"))
    if last_successful is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful)) | 
            ((F.col(ts_col) == F.lit(last_successful)) & (F.col(pk_col) > F.lit(last_successful_pk)))
        )
    rows_to_load = (
        rows_to_load
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
    .withColumn("bronze_ingested_ts", F.current_timestamp())
    .withColumn ("bronze_source_table", F.lit(source_table))
                 )
    rows_count = rows_to_load.count()
    print(f"rows to load: {rows_count}")
    if rows_count == 0:
        print(f"no new data for table {table_name}.")
        continue
    
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)
    
    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
    max_pk = (rows_to_load
        .filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(pk_col).cast("long").alias("max_pk"))
        .collect()[0]["max_pk"])
    
    upsert_bronze_control(table_name, ts_col, pk_col, max_ts, max_pk, rows_count, bronze_run_id)
    print(f"wrote {rows_count} rows to {target_table}")

        
        
        

In [0]:
print("orders Bronze count :", spark.sql("select count(*) from 00_bronze.orders_raw").collect()[0][0])
print("products Bronze count :", spark.sql("select count(*) from 00_bronze.products_raw").collect()[0][0])
print("payments Bronze count :", spark.sql("select count(*) from 00_bronze.payments_raw").collect()[0][0])

display(spark.sql("select * from 00_bronze.ingestion_control").orderBy("table_name"))
